In [ ]:
# Cell 1 — Setup environment & imports
!pip install -q transformers datasets jiwer accelerate

import os
os.environ["XLA_USE_BF16"] = "1"
os.environ["WANDB_DISABLED"] = "true"

import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp

from datasets import load_dataset, Audio, load_metric
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union

print("PyTorch/XLA device (example):", xm.xla_device())

In [ ]:
# Cell 2 — Config & dataset loading
MODEL_NAME = "openai/whisper-small"
LANGUAGE = "en"
DATASET_NAME = "common_voice"
SAMPLE_RATE = 16000

# For quick tests before full training:
MAX_TRAIN_SAMPLES = None  # set to e.g. 200 for quick smoke test
MAX_EVAL_SAMPLES = None   # set to e.g. 50 for quick smoke test

print("Loading dataset...")
common_voice = load_dataset(DATASET_NAME, LANGUAGE)
print(common_voice)

In [ ]:
# Cell 3 — Preprocessing
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
wer_metric = load_metric("wer")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # Common Voice uses 'sentence' column for transcription text
    text_field = "sentence" if "sentence" in batch else ("text" if "text" in batch else None)
    if text_field is None:
        raise ValueError("Couldn't find text field in dataset sample; ensure dataset has 'sentence' or 'text' column.")
    batch["labels"] = processor.tokenizer(batch[text_field]).input_ids
    return batch

# Ensure audio column uses requested sampling rate
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

if MAX_TRAIN_SAMPLES:
    common_voice["train"] = common_voice["train"].select(range(min(len(common_voice["train"]), MAX_TRAIN_SAMPLES)))
if MAX_EVAL_SAMPLES and "test" in common_voice:
    common_voice["test"] = common_voice["test"].select(range(min(len(common_voice["test"]), MAX_EVAL_SAMPLES)))

# Map preprocessing (non-batched to be memory safe)
common_voice = common_voice.map(prepare_dataset, remove_columns=[c for c in common_voice["train"].column_names if c not in ["audio","sentence","text","input_features","labels"]], num_proc=1)
print("Preprocessing complete. Example:", common_voice["train"][0])

In [ ]:
# Cell 4 — Data collator & metrics
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [f["input_features"] for f in features]
        label_features = [f["labels"] for f in features]

        batch = self.processor.feature_extractor.pad({"input_features": input_features}, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad({"input_ids": label_features}, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100)
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # replace -100 in labels as tokenizer.pad_token_id for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator and metrics ready.")

In [ ]:
# Cell 5 — TPU training function (this will be spawned across TPU cores)
def _mp_fn(index):
    device = xm.xla_device()
    print(f"Process {index} running on device {device}")

    # Load model & move to device
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model.to(device)

    training_args = Seq2SeqTrainingArguments(
        output_dir="./whisper-tpu-output",
        per_device_train_batch_size=4,  # per TPU core; reduce if OOM
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=1,
        learning_rate=1e-5,
        warmup_steps=50,
        max_steps=500,  # change for full training
        logging_steps=25,
        evaluation_strategy="steps",
        eval_steps=250,
        save_steps=500,
        predict_with_generate=True,
        generation_max_length=225,
        fp16=False,
        bf16=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        load_best_model_at_end=True,
        report_to=["none"]
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=common_voice["train"],
        eval_dataset=common_voice["test"] if "test" in common_voice else None,
        data_collator=data_collator,
        tokenizer=processor.tokenizer,
        compute_metrics=compute_metrics if "test" in common_voice else None,
    )

    trainer.train()

    if xm.is_master_ordinal():
        trainer.save_model("./whisper-tpu-output")
        print("Master process saved the model.")

In [ ]:
# Cell 6 — Launch training across TPU cores (8 processes)
# WARNING: this will start training across TPU cores. For a smoke test reduce max_steps and dataset sizes above.
xmp.spawn(_mp_fn, args=(), nprocs=8, start_method="fork")